# Argo — catalog, the `profile:` selector, and a T–S diagram

The [quickstart](quickstart.ipynb) covered a region fetch and a temperature–depth
profile. This notebook goes one level deeper into the `earthlens.argo` API: the
**parameter catalog** (how to discover the `phy` / `bgc` vocabularies), the
**`profile:` selector** (one float, one cycle), and a **temperature–salinity (T–S)
diagram** built from one float's profiles. Argo is open data — no credentials;
this backend needs `pip install earthlens[argo]`.

## Setup

`Catalog` is the bundled parameter vocabulary; `EarthLens` is the unified entry
point. We use `source="gdac"` (the Global Data Assembly Centre) throughout.

In [ ]:
import matplotlib.pyplot as plt

from earthlens.earthlens import EarthLens
from earthlens.argo import Catalog

## 1 — The parameter catalog

Argo measurements come in two `argopy` dataset families, selected with `dataset=`:
`"phy"` (core physical) and `"bgc"` (biogeochemical). The bundled `Catalog`
enumerates the canonical parameter names per family — useful for discovering the
BGC vocabulary and for the did-you-mean validation a region request runs.

In [ ]:
cat = Catalog()
print('families:', sorted(cat.datasets))
print('phy parameters:', sorted(cat.parameters_for('phy')))
print('bgc parameters:', sorted(cat.parameters_for('bgc')))

A region request validates the names you pass against the chosen family (a typo
raises a `ValueError` with the closest match), but it does not subset the returned
columns — `argopy` returns the whole family.

## 2 — One profile with the `profile:` selector

A `profile:<WMO>/<cycle>` token targets a single float's single cycle. Here we pull
cycle 12 of float `6902746` and plot its temperature and salinity against pressure
(≈ depth), y-axis inverted so the surface is at the top.

In [ ]:
one_profile = EarthLens(
    data_source="argo",
    variables=["profile:6902746/12"],
    start="2018-01-01",
    end="2021-12-31",
    path="_nb_out",
    source="gdac",
).download(progress_bar=False)

print(one_profile.shape, "| cycle(s):", sorted(one_profile["CYCLE_NUMBER"].unique()))
one_profile[["PLATFORM_NUMBER", "CYCLE_NUMBER", "PRES", "TEMP", "PSAL"]].head()

In [ ]:
fig, (axt, axs) = plt.subplots(1, 2, figsize=(7, 5), sharey=True)
axt.plot(one_profile["TEMP"], one_profile["PRES"], marker=".", lw=1)
axt.set_xlabel("Temperature (°C)"); axt.set_ylabel("Pressure (dbar)")
axs.plot(one_profile["PSAL"], one_profile["PRES"], marker=".", lw=1, color="C1")
axs.set_xlabel("Salinity (psu)")
axt.invert_yaxis()
fig.suptitle("Argo float 6902746, cycle 12")
plt.tight_layout()

## 3 — A temperature–salinity (T–S) diagram

Plotting salinity against temperature across all of a float's profiles gives a T–S
diagram — a classic way to identify water masses. We pull every profile from float
`6902746` with a `float:` selector and colour each point by pressure.

In [ ]:
one_float = EarthLens(
    data_source="argo",
    variables=["float:6902746"],
    start="2018-01-01",
    end="2021-12-31",
    path="_nb_out",
    source="gdac",
).download(progress_bar=False)

print("profiles:", one_float["CYCLE_NUMBER"].nunique(), "| levels:", len(one_float))

In [ ]:
ax = one_float.plot.scatter(x="PSAL", y="TEMP", c="PRES", cmap="viridis_r", s=6, figsize=(6, 5))
ax.set_xlabel("Salinity (psu)"); ax.set_ylabel("Temperature (°C)")
ax.set_title("T–S diagram — Argo float 6902746")
plt.tight_layout()

## Takeaway

- `Catalog().parameters_for("phy" | "bgc")` is how you discover the valid parameter
  names per family (the BGC vocabulary in particular).
- `variables=["profile:<WMO>/<cycle>"]` fetches one float-cycle; `variables=["float:<WMO>"]`
  fetches every profile from a float — both ignore the bbox.
- The returned long-format `DataFrame` carries one row per measured level, so it
  plots straight into depth profiles and T–S diagrams.